# Market Dimensionality / Opportunity Set — Feasibility Test v0.2

**Objetivo:** testar se medidas simples e espectrais da estrutura de dependência entre ativos contêm informação útil sobre o *opportunity set* cross-sectional.

O **Effective Rank (ER)** continua como nosso candidato principal, mas **não será tratado como protagonista por definição**. Ele terá que demonstrar utilidade incremental frente a benchmarks estruturalmente mais simples.

Este notebook **não otimiza Sharpe, CAGR ou parâmetros**. Ele é um teste de falsificação e comparação de representações.

## Research question

> A estrutura de dependência do mercado contém informação estável sobre o ambiente em que estratégias cross-sectional tendem a funcionar melhor ou pior?

## Hipóteses congeladas nesta etapa

- **H1 — Sanity estrutural:** ER baixo deve estar associado a maior concentração do movimento do mercado em poucos componentes.
- **H2 — Redundância estrutural:** ER, correlação média e domínio do primeiro componente provavelmente capturam aspectos semelhantes; isso deve ser quantificado, não ignorado.
- **H3 — Informação incremental (a testar depois):** ER só merece permanecer como medida principal se explicar a eficácia futura do alpha de forma mais estável ou informativa que benchmarks simples.
- **H4 — Robustez temporal (a testar depois):** qualquer relação com alpha deve sobreviver fora da amostra e a escolhas razoáveis de janela.

> Nesta versão, **nenhuma dessas medidas recebe status de “melhor”**. O teste de alpha será a horse race que decide isso.

## Universo inicial de feasibility

ETFs setoriais americanos líquidos e com histórico longo:

`XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, XLY`

Esta escolha é **apenas para feasibility**; não implica que o modelo final usará ETFs ou mercado americano.

## 0. Regras metodológicas pré-definidas

1. Somente informação disponível até a data \(t\) pode entrar em qualquer variável calculada em \(t\).
2. Nenhum parâmetro será escolhido com base em performance futura.
3. A janela-base estrutural será **252 pregões**.
4. Janelas de robustez, se necessárias, serão apenas **126 e 504 pregões**.
5. As estruturas serão estimadas com retornos diários e observadas em frequência mensal.
6. Preços serão ajustados por proventos/splits via `auto_adjust=True`.
7. Os dados baixados serão armazenados localmente para reprodutibilidade.
8. **Effective Rank, mean correlation e market mode serão comparados lado a lado.**
9. A complexidade do ER só será justificada se houver evidência de ganho informacional além das medidas simples.
10. O teste de alpha só começa após esta etapa estrutural.

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError(
        "Instale yfinance antes de rodar este notebook: pip install yfinance"
    ) from exc

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ["XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY"]
START = "2000-01-01"
END = None

WINDOW = 252
MIN_COVERAGE = 0.98
CACHE_FILE = DATA_DIR / "us_sector_etfs_adjusted_close.csv"

TICKERS

## 1. Coleta e cache dos dados

O notebook tenta reutilizar um CSV local antes de acessar a internet. Isso permite repetir exatamente a mesma análise posteriormente.

In [ ]:
def load_prices(force_download: bool = False) -> pd.DataFrame:
    if CACHE_FILE.exists() and not force_download:
        return pd.read_csv(CACHE_FILE, index_col=0, parse_dates=True).sort_index()

    raw = yf.download(
        TICKERS,
        start=START,
        end=END,
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=True,
    )

    if raw.empty:
        raise RuntimeError("Download retornou vazio.")

    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            prices = raw["Close"].copy()
        elif "Close" in raw.columns.get_level_values(1):
            prices = raw.xs("Close", axis=1, level=1).copy()
        else:
            raise KeyError("Coluna Close não encontrada.")
    else:
        prices = raw[["Close"]].rename(columns={"Close": TICKERS[0]})

    prices = prices.reindex(columns=TICKERS).sort_index()
    prices.to_csv(CACHE_FILE)
    return prices

prices = load_prices()
prices.tail()

## 2. Auditoria mínima de qualidade

Não fazemos *backfill*. Pequenos buracos isolados podem ser preenchidos com o último preço conhecido, mas períodos longos incompletos são descartados.

In [ ]:
coverage = prices.notna().mean().sort_values()
display(coverage.to_frame("coverage"))

first_valid_dates = prices.apply(pd.Series.first_valid_index)
common_start = max(first_valid_dates)

px = prices.loc[common_start:].copy()
daily_coverage = px.notna().mean(axis=1)
px = px.loc[daily_coverage >= MIN_COVERAGE]
px = px.ffill(limit=3)
px = px.dropna(how="any")

print("Common start:", common_start)
print("Observações:", len(px))
print("Início:", px.index.min())
print("Fim:", px.index.max())

assert not px.empty
assert not px.isna().any().any()

## 3. Retornos

Usamos retornos logarítmicos diários:

\[
r_{i,t} = \ln(P_{i,t}/P_{i,t-1})
\]

In [ ]:
returns = np.log(px / px.shift(1)).dropna()
returns.describe().T[["mean", "std", "min", "max"]]

## 4. Definição do Effective Rank

Para uma matriz de correlação \(C_t\), com autovalores \(\lambda_1,\ldots,\lambda_N\):

\[
ER_t =
\frac{\left(\sum_i \lambda_i\right)^2}
{\sum_i \lambda_i^2}
\]

Também calculamos:

\[
MarketMode_t =
\frac{\lambda_{1,t}}{\sum_i \lambda_{i,t}}
\]

e a correlação média entre pares de ativos.

In [ ]:
def effective_rank_from_corr(corr: np.ndarray) -> tuple[float, float]:
    eigvals = np.linalg.eigvalsh(corr)
    eigvals = np.clip(eigvals, 0.0, None)
    eigvals = np.sort(eigvals)[::-1]

    total = eigvals.sum()
    if total <= 0:
        return np.nan, np.nan

    er = (total ** 2) / np.square(eigvals).sum()
    market_mode = eigvals[0] / total
    return float(er), float(market_mode)


def mean_pairwise_corr(corr: np.ndarray) -> float:
    upper = corr[np.triu_indices(corr.shape[0], k=1)]
    return float(np.nanmean(upper))


def rolling_structure(rets: pd.DataFrame, window: int = 252) -> pd.DataFrame:
    rows = []

    for end in range(window, len(rets) + 1):
        window_rets = rets.iloc[end - window:end]
        corr = window_rets.corr().to_numpy()

        er, market_mode = effective_rank_from_corr(corr)

        rows.append({
            "date": rets.index[end - 1],
            "effective_rank": er,
            "market_mode": market_mode,
            "mean_corr": mean_pairwise_corr(corr),
        })

    return pd.DataFrame(rows).set_index("date")

structure_daily = rolling_structure(returns, WINDOW)
structure_daily.tail()

## 5. Frequência mensal da análise estrutural

O ER e os benchmarks são estimados diariamente com janela *rolling*, mas avaliamos sua informação usando apenas a **última observação realmente disponível em cada mês**.

O timestamp original é preservado. Assim, se a última observação disponível de agosto for 10/08, ela continuará aparecendo como 10/08 — nunca como 31/08.

In [ ]:
# Usamos a última observação REAL disponível em cada mês e preservamos
# o timestamp verdadeiro dessa observação.
#
# Evita o problema do resample("ME"), que pode rotular 10/08 como 31/08
# mesmo sem existir dado de 31/08.
month_key = structure_daily.index.to_period("M")
structure_monthly = (
    structure_daily
    .groupby(month_key, group_keys=False)
    .tail(1)
    .dropna()
)

structure_monthly.tail()

# TESTE H1 — Sanity estrutural

Esperamos:

\[
ER \downarrow \Rightarrow MarketMode \uparrow
\]

e, em geral:

\[
ER \downarrow \Rightarrow CorrMedia \uparrow
\]

**Importante:** isso é um *sanity check*, não evidência econômica independente.

Como o ER é função dos autovalores da própria matriz de correlação, uma associação forte com medidas de correlação/concentração é parcialmente mecânica. O objetivo aqui é confirmar que a implementação e a interpretação estrutural são coerentes.

In [ ]:
corr_summary = structure_monthly[
    ["effective_rank", "market_mode", "mean_corr"]
].corr()

corr_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    structure_monthly["effective_rank"],
    structure_monthly["market_mode"],
    alpha=0.7,
)
ax.set_xlabel("Effective Rank")
ax.set_ylabel("Share do 1º componente (Market Mode)")
ax.set_title("Effective Rank vs. concentração no 1º componente")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    structure_monthly["effective_rank"],
    structure_monthly["mean_corr"],
    alpha=0.7,
)
ax.set_xlabel("Effective Rank")
ax.set_ylabel("Correlação média entre ativos")
ax.set_title("Effective Rank vs. correlação média")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(structure_monthly.index, structure_monthly["effective_rank"])
ax.set_xlabel("Data")
ax.set_ylabel("Effective Rank")
ax.set_title("Effective Rank ao longo do tempo")
plt.show()

## 6. Separação estrutural por quintis de ER

Ainda **não testamos retorno**.

Este bloco apenas verifica se extremos de ER correspondem a estados de dependência distintos. A monotonicidade aqui não será tratada como prova de previsibilidade; ela serve para documentar a representação do mercado.

In [ ]:
tmp = structure_monthly.copy()
tmp["er_quintile"] = pd.qcut(
    tmp["effective_rank"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    duplicates="drop",
)

structural_by_q = (
    tmp.groupby("er_quintile", observed=True)
       .agg(
           n=("effective_rank", "size"),
           er_mean=("effective_rank", "mean"),
           mean_corr=("mean_corr", "mean"),
           market_mode=("market_mode", "mean"),
       )
)

structural_by_q

## 7. Registro de H1

A partir dos resultados da v0.1, a classificação correta é:

- **Implementação do ER:** PASS
- **Interpretação estrutural:** PASS
- **Separação de estados de dependência:** PASS
- **Evidência de previsibilidade:** NÃO TESTADA
- **Informação incremental do ER:** NÃO TESTADA
- **Justificativa para timing/alocação:** NÃO TESTADA

Ou seja: H1 é um **sanity check aprovado**, não uma validação da estratégia.

In [ ]:
def structural_diagnostics(structure: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "corr_ER_market_mode": structure["effective_rank"].corr(structure["market_mode"]),
        "corr_ER_mean_corr": structure["effective_rank"].corr(structure["mean_corr"]),
        "ER_mean": structure["effective_rank"].mean(),
        "ER_std": structure["effective_rank"].std(),
        "ER_min": structure["effective_rank"].min(),
        "ER_max": structure["effective_rank"].max(),
        "n_months": len(structure),
    })

diagnostics = structural_diagnostics(structure_monthly)
diagnostics

# H2 — Structural Benchmark / Horse Race

A tese central agora é **Opportunity Set**, não “Effective Rank Strategy”.

Congelamos três descritores do mesmo ambiente:

### 1. Effective Rank
\[
ER_t =
\frac{(\sum_i\lambda_i)^2}{\sum_i\lambda_i^2}
\]

### 2. Mean Correlation
\[
\overline{\rho}_t
=
\frac{2}{N(N-1)}
\sum_{i<j}\rho_{ij,t}
\]

### 3. Market Mode
\[
MM_t =
\frac{\lambda_{1,t}}{\sum_i\lambda_i}
\]

No teste de alpha, os três serão avaliados lado a lado.

**Regra pré-registrada:** o ER só será promovido a medida principal se apresentar vantagem clara de interpretação, estabilidade OOS ou poder condicional em relação aos benchmarks simples.

In [ ]:
# Comparação de redundância estrutural.
# Pearson mostra relação linear; Spearman (via ranks) mostra relação monotônica.

structural_measures = structure_monthly[
    ["effective_rank", "mean_corr", "market_mode"]
].copy()

pearson_corr = structural_measures.corr(method="pearson")
spearman_corr = structural_measures.rank().corr(method="pearson")

print("Pearson:")
display(pearson_corr)

print("Spearman:")
display(spearman_corr)

In [ ]:
# Padronização apenas para visualização estrutural.
# Nenhum dado futuro é usado para sinal nesta etapa, pois não há sinal de trading.

z = (structural_measures - structural_measures.mean()) / structural_measures.std()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(z.index, z["effective_rank"], label="Effective Rank")
ax.plot(z.index, -z["mean_corr"], label="- Mean Correlation")
ax.plot(z.index, -z["market_mode"], label="- Market Mode")
ax.set_title("Descritores estruturais padronizados")
ax.set_xlabel("Data")
ax.set_ylabel("Z-score (visualização)")
ax.legend()
plt.show()

## Como interpretar este bloco

Se as três séries forem quase idênticas, isso **não mata a tese Opportunity Set**. Apenas significa que o ER pode não ser necessário.

O teste decisivo será:

\[
StructuralMetric_t
\rightarrow
FutureAlphaQuality_{t+1}
\]

A horse race futura comparará:

- separação por quantis;
- relação contínua;
- estabilidade por subperíodo;
- comportamento OOS;
- robustez a janela;
- utilidade econômica após custos.

Não escolheremos a “melhor métrica” olhando apenas o período de desenvolvimento.

# PONTO DE DECISÃO — Definição do alpha experimental

Agora a próxima escolha material é **qual alpha usar como instrumento de teste do Opportunity Set**.

### Opção A — Residual Momentum
Testa persistência de movimentos específicos após retirar um componente comum.

**Prós**
- ligação conceitual forte com dimensionalidade;
- reduz a chance de confundirmos stock selection com simples beta comum;
- permite perguntar se ambientes mais dimensionais favorecem informação idiossincrática.

**Contras**
- exige uma regra explícita para estimar/remover o fator comum;
- abre novas decisões: benchmark de mercado, janela de beta e construção do residual.

### Opção B — Momentum cross-sectional simples
Rankeia os ETFs pelo retorno passado.

**Prós**
- extremamente transparente;
- excelente controle experimental;
- quase nenhuma camada estatística adicional.

**Contras**
- pode misturar movimento relativo com componentes comuns de mercado;
- menos alinhado à interpretação de dimensionalidade.

### Recomendação atual
Usar **Momentum simples como controle** e **Residual Momentum como alpha principal**.

Porém, antes de implementar A precisamos congelar **como residualizar**. Essa é a próxima decisão relevante.

## Registro de decisões — v0.2

| Item | Decisão |
|---|---|
| Tese central | Market Dimensionality / Opportunity Set |
| Universo inicial | ETFs setoriais EUA |
| Frequência dos dados | diária |
| Frequência da análise estrutural | mensal, preservando data real |
| Janela estrutural baseline | 252 pregões |
| Candidato principal | Effective Rank |
| Benchmarks estruturais | Mean Correlation; Market Mode |
| Status H1 | PASS como sanity check |
| Informação incremental ER | **NÃO TESTADA** |
| Alpha-base | **PENDENTE** |
| OOS | **a definir antes do primeiro teste de alpha** |
| Estratégia de alocação | **não definida nesta fase** |

### Regra de governança
Nenhuma medida estrutural será escolhida por ter o melhor resultado in-sample.  
A decisão final deverá considerar simplicidade, estabilidade e evidência OOS.